# EDA 06 — Statistiques de Criminalite (SSMSI)
**Source** : SSMSI / data.gouv.fr | **Bronze** : `data/bronze/crime/`

In [ ]:

import os, glob, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
warnings.filterwarnings("ignore")

PALETTE = {
    "primary":   "#1565C0",
    "secondary": "#E53935",
    "accent":    "#2E7D32",
    "neutral":   "#546E7A",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.framealpha": 0.9,
    "legend.fontsize": 10,
})

BRONZE_ROOT = os.path.abspath("../../data/bronze")
NB_DIR      = os.path.abspath(".")
FIGURES_DIR = os.path.join(NB_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(name, source_prefix, tight=True):
    dest = os.path.join(FIGURES_DIR, source_prefix)
    os.makedirs(dest, exist_ok=True)
    path = os.path.join(dest, f"{name}.png")
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  Saved -> figures/{source_prefix}/{name}.png")

print("Setup OK — figures ->", FIGURES_DIR)


In [ ]:

def draw_schema(title, columns, source_prefix, filename="schema"):
    n_rows = len(columns)
    fig_h  = max(3.0, 0.48 * n_rows + 1.6)
    fig, ax = plt.subplots(figsize=(13, fig_h))
    ax.axis("off")
    fig.suptitle(title, fontsize=14, fontweight="bold", y=0.99, ha="center")
    headers = ["Colonne", "Type", "Description"]
    col_x   = [0.0, 0.22, 0.37]
    col_w   = [0.22, 0.15, 0.63]
    row_h   = 1.0 / (n_rows + 1)
    for i, (hdr, x, w) in enumerate(zip(headers, col_x, col_w)):
        rect = mpatches.FancyBboxPatch(
            (x, 1 - row_h), w - 0.006, row_h * 0.88,
            boxstyle="round,pad=0.01", linewidth=0.5,
            edgecolor="#90A4AE", facecolor="#1565C0",
            transform=ax.transAxes, clip_on=False)
        ax.add_patch(rect)
        ax.text(x + w/2, 1 - row_h/2, hdr,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=11, fontweight="bold", color="white")
    type_colors = {
        "str":"#1565C0","int":"#2E7D32","float":"#6A1B9A",
        "datetime":"#E65100","bool":"#AD1457","date":"#00695C",
    }
    for ridx, (col_name, col_type, col_desc) in enumerate(columns):
        y_top = 1 - (ridx + 2) * row_h
        bg = "#F5F7FA" if ridx % 2 == 0 else "white"
        for x, w in zip(col_x, col_w):
            rect = mpatches.FancyBboxPatch(
                (x, y_top), w - 0.006, row_h * 0.88,
                boxstyle="round,pad=0.01", linewidth=0.5,
                edgecolor="#CFD8DC", facecolor=bg,
                transform=ax.transAxes, clip_on=False)
            ax.add_patch(rect)
        y_mid = y_top + row_h * 0.44
        tc = type_colors.get(col_type.split("[")[0].lower(), "#37474F")
        ax.text(col_x[0]+col_w[0]/2, y_mid, col_name,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, fontweight="bold", color="#1A237E")
        ax.text(col_x[1]+col_w[1]/2, y_mid, col_type,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=9, fontfamily="monospace", color=tc)
        ax.text(col_x[2]+0.01, y_mid, col_desc,
                transform=ax.transAxes, ha="left", va="center",
                fontsize=9.5, color="#37474F")
    plt.subplots_adjust(top=0.92, bottom=0)
    save_fig(filename, source_prefix)
    plt.show()


In [ ]:
PREFIX = "06_crime"
draw_schema(
    "Bronze Schema — Criminalite SSMSI (base communale)",
    [
        ("commune_code",   "str",      "Code INSEE commune (ex: 75112 pour le 12e)"),
        ("arrondissement", "int",      "Numero arrondissement Paris (1-20)"),
        ("crime_category", "str",      "Categorie SSMSI (ex: Cambriolages de logement)"),
        ("crime_count",    "int",      "Nombre de faits enregistres"),
        ("rate_per_1000",  "float",    "Taux de faits pour 1 000 habitants"),
        ("year",           "int",      "Annee de reference (2016 -> derniere disponible)"),
        ("ingested_at",    "datetime", "Horodatage UTC d'ingestion"),
    ], PREFIX
)

In [ ]:
crime_files = sorted(glob.glob(f"{BRONZE_ROOT}/crime/**/*.parquet", recursive=True))
if crime_files:
    df = pd.concat([pd.read_parquet(f) for f in crime_files], ignore_index=True)
else:
    rng = np.random.default_rng(42)
    cats = ["Cambriolages de logement","Vols sans violence contre des personnes",
            "Coups et blessures volontaires","Vols de vehicules","Trafic de stupefiants","Destructions et degradations"]
    rows = []
    base = {a:max(5,50-a*1.5+rng.normal(0,5)) for a in range(1,21)}
    for a in range(1,21):
        for y in range(2016,2025):
            for c in cats:
                rate = base[a]*rng.uniform(0.7,1.3)
                rows.append({"commune_code":f"751{a:02d}","arrondissement":a,
                             "crime_category":c,"crime_count":int(rate*200),"rate_per_1000":rate,"year":y})
    df = pd.DataFrame(rows)
latest = df["year"].max(); df_latest = df[df["year"]==latest]
print(f"Shape : {df.shape}  |  Annee recente : {latest}")

In [ ]:
cat_total = df_latest.groupby("crime_category")["crime_count"].sum().sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(12,7))
colors = plt.cm.Reds(np.linspace(0.35,0.9,len(cat_total)))
bars = ax.barh(cat_total.index, cat_total.values, color=colors, edgecolor="white")
for bar, val in zip(bars, cat_total.values):
    ax.text(val+40,bar.get_y()+bar.get_height()/2,f"{val:,}",va="center",fontsize=10)
ax.set_title(f"Faits criminels par categorie — Paris {latest}"); ax.set_xlabel("Nombre de faits enregistres"); ax.grid(axis="x")
save_fig("crimes_par_categorie", PREFIX); plt.show()

In [ ]:
arr_rate = df_latest.groupby("arrondissement")["rate_per_1000"].sum().sort_values(ascending=False)
norm = (arr_rate-arr_rate.min())/(arr_rate.max()-arr_rate.min())
fig, ax = plt.subplots(figsize=(14,6))
ax.bar(arr_rate.index.astype(str),arr_rate.values,color=plt.cm.Reds(0.25+norm*0.7),edgecolor="white")
ax.set_xlabel("Arrondissement"); ax.set_ylabel("Taux cumule / 1 000 hab.")
ax.set_title(f"Taux de criminalite global par arrondissement — {latest}")
plt.setp(ax.xaxis.get_majorticklabels(),rotation=45)
sm = plt.cm.ScalarMappable(cmap="Reds",norm=plt.Normalize(arr_rate.min(),arr_rate.max()))
plt.colorbar(sm,ax=ax,label="Taux / 1 000 hab.")
save_fig("taux_criminalite_arrondissement", PREFIX); plt.show()

In [ ]:
top_cats = df.groupby("crime_category")["crime_count"].sum().nlargest(4).index
fig, ax = plt.subplots(figsize=(14,6))
line_colors = [PALETTE["primary"],PALETTE["secondary"],PALETTE["accent"],PALETTE["neutral"]]
for cat, col in zip(top_cats, line_colors):
    ts = df[df["crime_category"]==cat].groupby("year")["rate_per_1000"].mean()
    ax.plot(ts.index,ts.values,marker="o",linewidth=2.5,color=col,label=cat[:40])
ax.set_xlabel("Annee"); ax.set_ylabel("Taux moyen / 1 000 hab.")
ax.set_title("Evolution des 4 principales categories de crimes — Paris")
ax.legend(fontsize=9,loc="upper right"); ax.set_xticks(sorted(df["year"].unique()))
save_fig("evolution_criminalite", PREFIX); plt.show()